# One-region signal tracking and two-region coupling


This notebook tests whether `E` follows `external_input_ex_ex`, then uses a successful input-region configuration in a directed two-region simulation.


## Initialization



Load environment variables

<font color=#fda> NOTE: </font> Prior to loading any modules, we need to load the environment variables from the configuration files. This way, we can configure the TVB modules using environment variables.


In [ ]:
from dotenv import dotenv_values, find_dotenv
import os
import sys
from pathlib import Path


# Busqueda de ficheros dotenv
defaults_file = find_dotenv('.env_default', usecwd=True)

if not defaults_file:
    raise FileNotFoundError('Start Jupyter within the repository containing .env_default.')
env_dir = Path(defaults_file).resolve().parent

# Actualización de varaibles de entorno
settings = {**dotenv_values(defaults_file), **dotenv_values(env_dir / '.env')}
settings = {k: v if v is not None else '' for k, v in settings.items() }

os.environ.update(settings)

# Librerías auxiliares
import_dir = Path(settings.get('IMPORT_DIR') or env_dir).expanduser()
if not import_dir.is_absolute():
    import_dir = env_dir / import_dir
sys.path.insert(0, str(import_dir.resolve()))


Modules

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from dataclasses import asdict


In [ ]:
from tvb.datatypes.connectivity import Connectivity
from tvb.simulator.coupling import Linear
from tvb.simulator.monitors import Raw


In [ ]:
from aux.environment import resolve_directory
from aux.tvb.stimulus import build_stimulus
from aux.tvb.model import build_model
from aux.tvb.integrator import build_integrator
from aux.tvb.simulator import DynamicParameterSimulator, build_single_region_simulator
from aux.tvb.results import SimulationMetadata, ModelMetadata, ResultMetadata
from aux.tvb.plotting import plot_results


Console

In [ ]:
from rich.console import Console
pprint = Console(force_terminal=True, force_jupyter=False, width=140).print

pprint(f'The console is "ready" to print messages with rich formatting.')

Set main directory paths

In [ ]:

#ToDo: Cambiar el nombe de las variables de entorno
WORK_DIR = resolve_directory(settings['WORK_DIR'], env_dir)

os.makedirs(WORK_DIR, exist_ok=True)
pprint(f'Working directory: {WORK_DIR}')

## Simulation values


In [ ]:
# 1) Simulation outputs.
OUTPUT_DTYPE = "float32"
OUTPUT_STATES = ["E"]

STATE_UNITS = {
    "E": "kHz", "I": "kHz",
    "C_ee": "kHz²", "C_ei": "kHz²", "C_ii": "kHz²",
    "W_e": "pA", "W_i": "pA", "ou_drift": "N.A.",
}
STIMULUS_UNITS = {"external_input_ex_ex": "kHz"}


In [ ]:
# 1.1) Simulation parameters.
DURATION_MS = 1000.0
DT_MS = 0.5


In [ ]:
# 1.2) Input stimulus: repeated pulses expose both level and timing errors.
STIMULUS = {
    "external_input_ex_ex": {
        "pulse_train": {
            "baseline_khz": 0.005,
            "pulse_amplitude_khz": 0.04,
            "pulse_onset_ms": 200.0,
            "pulse_duration_ms": 160.0,
            "pulse_period_ms": 320.0,
        },
    },
}


In [ ]:
# 1.3) Parámetros del modelo Zerlaut-AdEx.
#MODEL_NAME = "ZerlautAdaptationFirstOrder"
#MODEL_NAME = "ZerlautAdaptationSecondOrder"

MODEL_PARAMETERS = {
    # Población y conectividad
    "N_tot": 10000,
    "p_connect_e": 0.05,
    "p_connect_i": 0.05,
    "g": 0.2,
    "K_ext_e": 400,
    "K_ext_i": 35,
    
    # Parametros de membrana
    "g_L": 10.0,
    "C_m": 200.0,
    "E_L_e": -60.0,
    "E_L_i": -65.0,

    # Parametors de adaptación
    "a_e": 4.0,
    "b_e": 60.0,
    "tau_w_e": 200.0,
    "a_i": 0.0,
    "b_i": 20.0,
    "tau_w_i": 100.0,

    # Parametros sinápticos
    "E_e": 0.0,
    "E_i": -80.0,
    "Q_e": 1.0,
    "Q_i": 5.0,
    "tau_e": 5.0,
    "tau_i": 5.0,
    
    "T": 10.0,

    # Parametro de segundo orden (no hay un motivo matemático)
    "S_i": 1, # Escala la entrada remota recibida por la población inhibidora en el modelo de segundo orden.

    # Parametros de ruido
    "weight_noise": 2e-4, #10.5
    "tau_OU": 2, #0.5
}

# Todos los parámetros son de tipo float menos lo que están en esta lista. 
INTEGER_PARAMETERS = {"N_tot", "K_ext_e", "K_ext_i"}


## Build the stimulus


In [ ]:
time_ms, stimulus = build_stimulus(STIMULUS, DURATION_MS, DT_MS)
input_signal = stimulus[0][1][:, 0]


## Deterministic Simulation


### Compare model parameters


In [ ]:
# These values include the 01b defaults and a focused search around lower external gain.
PARAMETER_CONFIGS = {
    "01b defaults": {},
    "Lower external gain": {"K_ext_e": 100},
    "Fast response": {"K_ext_e": 127, "Q_e": 0.9285591413880798, "T": 1.6506473208720385},
    "Tracking candidate": {"K_ext_e": 116, "Q_e": 0.9750688212874393, "T": 1.232085978666654},
}



In [ ]:
integrator = build_integrator(DT_MS, integrator_class="HeunDeterministic")
results = {}
for name, changes in PARAMETER_CONFIGS.items():
    parameters = MODEL_PARAMETERS | changes
    model = build_model(parameters, INTEGER_PARAMETERS,
                        model_class="ZerlautAdaptationFirstOrder",
                        variable_of_interest=OUTPUT_STATES)
    
    simulator = build_single_region_simulator(model, stimulus, integrator)
    time_sim, raw = simulator.run(n_steps=len(time_ms))[0]
    results[name] = raw[:, :, 0, 0]
    pprint(f"{name}: {len(time_sim)} samples")


### Plot results


In [ ]:
metadata = ResultMetadata(
    simulation=SimulationMetadata(DURATION_MS, DT_MS, integrator, OUTPUT_DTYPE),
    model=ModelMetadata(name="ZerlautAdaptationFirstOrder", parameters=MODEL_PARAMETERS,
                        integer_parameters=list(INTEGER_PARAMETERS),
                        output_variables=OUTPUT_STATES, state_units=STATE_UNITS,
                        stimulus_units=STIMULUS_UNITS),
    stimulus=STIMULUS,
)
fig = plt.figure(figsize=(4.5 * len(results), 6), constrained_layout=True)
grid = fig.add_gridspec(1, len(results))
for column, (name, output_E) in enumerate(results.items()):
    _, axes = plot_results(time_sim, {"E": output_E}, metadata,
                           show_stimulus=True, ax=fig.add_subplot(grid[0, column]),
                           plot_states=["E"])
    axes[0].set_title(name)
plt.show()


### Explore excitatory adaptation



In [ ]:
ADAPTATION_CONFIGS = {
    "Half spike adaptation": {
        **PARAMETER_CONFIGS["Tracking candidate"], "K_ext_e": 95, "b_e": 30.0,
    },
    "Half adaptation": {
        **PARAMETER_CONFIGS["Tracking candidate"], "K_ext_e": 100,
        "a_e": 2.0, "b_e": 30.0,
    },
    "Reduced adaptation": {
        **PARAMETER_CONFIGS["Tracking candidate"], "K_ext_e": 95,
        "a_e": 0.0, "b_e": 30.0,
    },
    "Faster adaptation decay": {
        **PARAMETER_CONFIGS["Tracking candidate"], "K_ext_e": 95,
        "a_e": 0.0, "b_e": 30.0, "tau_w_e": 100.0,
    },
}


In [ ]:
adaptation_results = {}
for name, changes in ADAPTATION_CONFIGS.items():
    parameters = MODEL_PARAMETERS | changes
    model = build_model(parameters, INTEGER_PARAMETERS,
                        model_class="ZerlautAdaptationFirstOrder",
                        variable_of_interest=OUTPUT_STATES)
    simulator = build_single_region_simulator(model, stimulus, integrator)
    time_sim, raw = simulator.run(n_steps=len(time_ms))[0]
    adaptation_results[name] = raw[:, model.state_variables.index("E"), 0, 0]
    pprint(f"{name}: {len(time_sim)} samples")


### Plot adaptation results


In [ ]:
fig = plt.figure(figsize=(4.5 * len(adaptation_results), 6), constrained_layout=True)
grid = fig.add_gridspec(1, len(adaptation_results))
for column, (name, output_E) in enumerate(adaptation_results.items()):
    _, axes = plot_results(time_sim, {"E": output_E}, metadata,
                           show_stimulus=True, ax=fig.add_subplot(grid[0, column]),
                           plot_states=["E"])
    axes[0].set_title(name)
plt.show()


### Test the pulse-onset peaks

`external_input_ex_ex` drives `E` directly, while recurrent activity recruits `I`. This allows `E` to rise before inhibition catches up. In the Zerlaut fluctuation equations, `p_connect_i` scales recurrent inhibitory contacts. Increasing it strengthens that feedback but also lowers the sustained response. With `a_e = 0`, the stationary excitatory adaptation is `W_e = tau_w_e * b_e * E`. Solving the stationary `E` and `I` equations at the high input level gives `b_e` close to 7 when `p_connect_i` is 0.06. The full simulation checks the transient as well as the plateau.


In [ ]:
PEAK_CONFIGS = {
    "Reduced adaptation": ADAPTATION_CONFIGS["Reduced adaptation"],
    "Stronger recurrent inhibition": {
        **ADAPTATION_CONFIGS["Reduced adaptation"],
        "p_connect_i": 0.06, "b_e": 7.0,
    },
}


In [ ]:
peak_results = {}
for name, changes in PEAK_CONFIGS.items():
    parameters = MODEL_PARAMETERS | changes
    model = build_model(parameters, INTEGER_PARAMETERS,
                        model_class="ZerlautAdaptationFirstOrder",
                        variable_of_interest=OUTPUT_STATES)
    simulator = build_single_region_simulator(model, stimulus, integrator)
    time_sim, raw = simulator.run(n_steps=len(time_ms))[0]
    peak_results[name] = raw[:, model.state_variables.index("E"), 0, 0]
    pprint(f"{name}: {len(time_sim)} samples")


### Plot the pulse onset


In [ ]:
fig = plt.figure(figsize=(5.5 * len(peak_results), 8), constrained_layout=True)
grid = fig.add_gridspec(2, len(peak_results), height_ratios=[3, 1])
for column, (name, output_E) in enumerate(peak_results.items()):
    _, axes = plot_results(time_sim, {"E": output_E}, metadata,
                           show_stimulus=True, ax=fig.add_subplot(grid[0, column]),
                           plot_states=["E"])
    axes[0].set_title(name)

    zoom_ax = fig.add_subplot(grid[1, column])
    zoom_ax.stairs(input_signal, np.r_[time_ms, DURATION_MS], label="Input")
    zoom_ax.plot(time_sim, output_E, label="E")
    zoom_ax.set_xlim(STIMULUS["external_input_ex_ex"]["pulse_train"]["pulse_onset_ms"] - 5,
                     STIMULUS["external_input_ex_ex"]["pulse_train"]["pulse_onset_ms"] + 20)
    zoom_ax.set_xlabel("Time (ms)")
    zoom_ax.set_ylabel("kHz")
    zoom_ax.grid(alpha=0.25)
    zoom_ax.legend()
plt.show()


## Two-region Simulation


No lo entiendo

$$
\begin{aligned}
f_e^{(r)} =& (\nu_e+10^{-6})(1-g)p_eN_{\mathrm{tot}} + \nu_e^{\mathrm{ext},(r)}K_e^{\mathrm{ext}},\\
f_i^{(r)} =& (\nu_i+10^{-6})gp_iN_{\mathrm{tot}}+\nu_i^{\mathrm{ext},(r)}K_i^{\mathrm{ext}},
\qquad r\in\{e,i\}.
\end{aligned}
$$


$$
\begin{aligned}
\nu_e^{\mathrm{ext},(r)} =& \boxed{c_0} + k_{\mathrm{local}}\nu_e + w_{\mathrm{noise}}x_{\mathrm{OU}} + \boxed{\nu^{\mathrm{drive}}_{e\to(r)}}\\
\nu_i^{\mathrm{ext},(r)} =& k_{\mathrm{local}}\nu_i + \boxed{\nu^{\mathrm{drive}}_{i\to(r)}}.
\end{aligned}
$$

In [ ]:
def build_two_region_simulator(input_parameters, input_signal, integrator):
    model = build_model(MODEL_PARAMETERS, INTEGER_PARAMETERS,
                        model_class="ZerlautAdaptationFirstOrder",
                        variable_of_interest=OUTPUT_STATES)
    
    for name, value in input_parameters.items():
        dtype = np.int64 if name in INTEGER_PARAMETERS else np.float64
        setattr(model, name, np.array([value, MODEL_PARAMETERS[name]], dtype=dtype))

    dynamic_parameters = [
        ("external_input_ex_ex", np.column_stack([input_signal, np.zeros_like(input_signal)]))
    ]
    connectivity = Connectivity(
        weights=np.array([[0.0, 0.0], [1.0, 0.0]]),
        tract_lengths=np.zeros((2, 2)), # distancia 0, delay 0
        speed=np.array([3.0]),
        centres=np.zeros((2, 3)),
        region_labels=np.array(["input", "output"]),
    )
    return DynamicParameterSimulator(
        model=model, dynamic_parameters=dynamic_parameters,
        connectivity=connectivity, coupling=Linear(a=np.array([1.0])),
        integrator=integrator, monitors=(Raw(),),
        initial_conditions=np.zeros((1, len(model.state_variables), 2, 1)),
    ).configure()


In [ ]:
input_parameters = ADAPTATION_CONFIGS["Reduced adaptation"]
two_region_simulator = build_two_region_simulator(input_parameters, input_signal, integrator)
time_two, raw_two = two_region_simulator.run(n_steps=len(time_ms))[0]
input_E = raw_two[:, :, 0, 0]
output_E = raw_two[:, :, 1, 0]

pprint("Connection weights:", two_region_simulator.connectivity.weights)
pprint("Connection delays (ms):", two_region_simulator.connectivity.delays)


### Plot results


In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(11, 8), sharex=True, constrained_layout=True)
axes[0].stairs(input_signal, np.r_[time_ms, DURATION_MS], label="external_input_ex_ex")
axes[1].plot(time_two, input_E, label="Input E")
axes[2].plot(time_two, output_E, label="Output E")
for axis in axes:
    axis.grid(alpha=0.25)
    axis.legend()
    axis.set_ylabel("kHz")
axes[-1].set_xlabel("Time (ms)")
plt.show()
